# Chapter 56: Autoencoders, Transfer Learning, and Generative Models

Build a transparent linear autoencoder for synthetic NRG package signals.


In [ ]:
from pathlib import Path
import sys,numpy as np,matplotlib.pyplot as plt
sys.path.insert(0,str(Path.cwd().parents[1]/'src'))
from datasciencebook.generative import *
print('Imports ready.')


Imports ready.


In [ ]:
rng=np.random.default_rng(56);base=rng.normal(size=(60,2));x=np.column_stack([base[:,0],base[:,1],base[:,0]+base[:,1]+rng.normal(0,.08,60)]);mean,components=fit_linear_autoencoder(x,2);z=encode(x,mean,components);recon=decode(z,mean,components);errors=reconstruction_error(x,recon)
print(f'Data={x.shape}; latent={z.shape}')
print(f'Mean reconstruction error={errors.mean():.6f}')


Data=(60, 3); latent=(60, 2)
Mean reconstruction error=0.000730


In [ ]:
unusual=np.array([[4.,-4.,4.]])
uerr=reconstruction_error(unusual,decode(encode(unusual,mean,components),mean,components))[0]
print(f'Routine 95th percentile={np.quantile(errors,.95):.6f}')
print(f'Unusual package error={uerr:.6f}')


Routine 95th percentile=0.001843
Unusual package error=1.783828


In [ ]:
path=interpolate_latent(z[0],z[1],5);decoded=decode(path,mean,components)
sample=reparameterize(np.zeros(2),np.zeros(2),np.array([.5,-.5]))
print('Latent sample:',sample.tolist())
print('Interpolation endpoints preserved:',bool(np.allclose(decoded[[0,-1]],recon[[0,1]])))


Latent sample: [0.5, -0.5]
Interpolation endpoints preserved: True


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(9,4));axes[0].scatter(z[:,0],z[:,1],c=errors,cmap='viridis');axes[0].plot(path[:,0],path[:,1],'r.-');axes[0].set(title='Latent representation',xlabel='z1',ylabel='z2');axes[1].hist(errors,bins=12);axes[1].axvline(uerr,color='r',label='unusual');axes[1].set(title='Reconstruction errors',xlabel='MSE');axes[1].legend();fig.tight_layout();plt.show()


## Interpretation

The two-dimensional subspace reconstructs routine correlated signals closely. The deliberately inconsistent package receives a much larger error, but a real threshold would require labeled validation.


In [ ]:
# Practice: fit a one-dimensional representation and compare errors.
